In [2]:
import random
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import faiss
from sentence_transformers import SentenceTransformer
import sklearn
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


# База знаний для домашней работы

In [25]:
# Тексты представляют собой учебный FAQ по Python

documents = [
    "Python — это язык программирования, который используется для веб-разработки, анализа данных и автоматизации.",
    "Списки в Python являются изменяемыми структурами данных и могут хранить несколько элементов.",
    "Словари хранят данные в формате ключ-значение.",
    "Функции в Python определяются с помощью ключевого слова def.",
    "Циклы позволяют выполнять действия несколько раз.",
    "Библиотека pandas используется для анализа данных.",
    "NumPy применяется для численных вычислений.",
    "Классы используются для объектно-ориентированного программирования.",
    "Обработка исключений выполняется с помощью конструкции try-except.",
    "Виртуальные окружения позволяют изолировать зависимости проекта."
]

print("Количество документов:", len(documents))

Количество документов: 10


In [26]:
for i in range(3):
    print(f"\nДокумент {i}:")
    print(documents[i])


Документ 0:
Python — это язык программирования, который используется для веб-разработки, анализа данных и автоматизации.

Документ 1:
Списки в Python являются изменяемыми структурами данных и могут хранить несколько элементов.

Документ 2:
Словари хранят данные в формате ключ-значение.


База знаний относится к теме Python. Это позволяет корректно тестировать retrieval и mini-RAG, потому что вся информация структурирована, ответы можно извлекать напрямую из текстов без сложной генерации, документы достаточно короткие

# Чанкинг

In [27]:
def chunk_text(text, chunk_size=20, overlap=5):
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

Выбранные параметры: 
1) chunk_size - длина фрагмента
2) overlap - пересечение, чтобы не терять контекст

In [28]:
all_chunks = []
chunk_sources = []

for i, doc in enumerate(documents):
    chunks = chunk_text(doc)
    for ch in chunks:
        all_chunks.append(ch)
        chunk_sources.append(i)

print("Всего чанков:", len(all_chunks))

Всего чанков: 10


In [29]:
for i in range(5):
    print(f"\nChunk {i}:")
    print(all_chunks[i])


Chunk 0:
Python — это язык программирования, который используется для веб-разработки, анализа данных и автоматизации.

Chunk 1:
Списки в Python являются изменяемыми структурами данных и могут хранить несколько элементов.

Chunk 2:
Словари хранят данные в формате ключ-значение.

Chunk 3:
Функции в Python определяются с помощью ключевого слова def.

Chunk 4:
Циклы позволяют выполнять действия несколько раз.


# Эмбеддинги и индекс `FAISS`

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
embeddings = model.encode(all_chunks, show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# FAISS индекс
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings))

print("Размер индекса:", index.ntotal)

Размер индекса: 10


In [32]:
def search(query, top_k=3):
    q_emb = model.encode([query])
    D, I = index.search(np.array(q_emb), top_k)
    
    results = []
    for idx in I[0]:
        results.append(all_chunks[idx])
    
    return results, I[0]

In [33]:
queries = [
    "Что такое Python?",
    "Как обрабатывать ошибки?",
    "Для чего используется NumPy?"
]

for q in queries:
    results, _ = search(q)
    print(f"\nЗапрос: {q}")
    for r in results:
        print("-", r)


Запрос: Что такое Python?
- Функции в Python определяются с помощью ключевого слова def.
- Списки в Python являются изменяемыми структурами данных и могут хранить несколько элементов.
- Python — это язык программирования, который используется для веб-разработки, анализа данных и автоматизации.

Запрос: Как обрабатывать ошибки?
- Обработка исключений выполняется с помощью конструкции try-except.
- Виртуальные окружения позволяют изолировать зависимости проекта.
- Словари хранят данные в формате ключ-значение.

Запрос: Для чего используется NumPy?
- NumPy применяется для численных вычислений.
- Списки в Python являются изменяемыми структурами данных и могут хранить несколько элементов.
- Python — это язык программирования, который используется для веб-разработки, анализа данных и автоматизации.


# Контрольные запросы и оценка retrieval

In [34]:
control_queries = [
    ("Что такое Python?", 0),
    ("Что такое список?", 1),
    ("Для чего нужен NumPy?", 6),
    ("Для чего используется pandas?", 5),
    ("Как объявить функцию?", 3)
]

In [46]:
k = 3

results_data = []
hits = 0
recalls = []

for query, true_doc in control_queries:
    _, idxs = search(query, top_k=k)
    retrieved_docs = [chunk_sources[i] for i in idxs]
    
    hit = int(true_doc in retrieved_docs)
    hits += hit
    
    recall = hit
    recalls.append(recall)
    
    results_data.append({
        "query": query,
        "expected_source": true_doc,
        "retrieved_sources": retrieved_docs,
        "hit_at_k": hit,
        "recall_at_k": recall
    })

hit_at_k = hits / len(control_queries)
recall_at_k = sum(recalls) / len(recalls)

print("Hit@k:", hit_at_k)
print("Recall@k:", recall_at_k)

df_eval = pd.DataFrame(results_data)
df_eval

Hit@k: 1.0
Recall@k: 1.0


,query,expected_source,retrieved_sources,hit_at_k,recall_at_k
0,Что такое Python?,0,"[3, 1, 0]",1,1
1,Что такое список?,1,"[1, 4, 8]",1,1
2,Для чего нужен NumPy?,6,"[6, 3, 1]",1,1
3,Для чего используется pandas?,5,"[5, 1, 4]",1,1
4,Как объявить функцию?,3,"[7, 2, 3]",1,1


# Небольшой эксперимент с параметрами retrieval

In [37]:
for k in [1, 3, 5]:
    hits = 0
    for query, true_doc in control_queries:
        _, idxs = search(query, top_k=k)
        retrieved_docs = [chunk_sources[i] for i in idxs]
        
        if true_doc in retrieved_docs:
            hits += 1
    
    print(f"Hit@{k}:", hits / len(control_queries))

Hit@1: 0.6
Hit@3: 1.0
Hit@5: 1.0


При k = 1 модель находит релевантный документ в 60% случаев, что говорит о том, что ближайший сосед не всегда является правильным. При увеличении k до 3 качество достигает 100%, то есть релевантный документ всегда попадает в топ-3 результатов. Дальнейшее увеличение k до 5 не дает прироста качества

# Обновление базы знаний и переиндексация

In [38]:
new_docs = [
    "Кортежи — это неизменяемые последовательности.",
    "Множества хранят только уникальные элементы."
]

documents_updated = documents + new_docs

In [39]:
all_chunks_new = []
chunk_sources_new = []

for i, doc in enumerate(documents_updated):
    chunks = chunk_text(doc)
    for ch in chunks:
        all_chunks_new.append(ch)
        chunk_sources_new.append(i)

embeddings_new = model.encode(all_chunks_new)

index_new = faiss.IndexFlatL2(embeddings_new.shape[1])
index_new.add(np.array(embeddings_new))

In [40]:
def search_new(query, top_k=3):
    q_emb = model.encode([query])
    D, I = index_new.search(np.array(q_emb), top_k)
    return I[0]

In [41]:
compare = []

for query, _ in control_queries:
    old = search(query)[1]
    new = search_new(query)
    
    compare.append({
        "query": query,
        "before_retrieved_sources": list(old),
        "after_retrieved_sources": list(new),
        "changed": old.tolist() != new.tolist()
    })

df_compare = pd.DataFrame(compare)
df_compare

,query,before_retrieved_sources,after_retrieved_sources,changed
0,Что такое Python?,"[3, 1, 0]","[3, 1, 0]",False
1,Что такое список?,"[1, 4, 8]","[1, 4, 11]",True
2,Для чего нужен NumPy?,"[6, 3, 1]","[6, 3, 1]",False
3,Для чего используется pandas?,"[5, 1, 4]","[5, 1, 4]",False
4,Как объявить функцию?,"[7, 2, 3]","[7, 2, 3]",False


# Mini-RAG

In [42]:
def mini_rag(query, top_k=3):
    results, idxs = search(query, top_k)
    
    context = " ".join(results)
    
    answer = f"Ответ на основе контекста: {context[:200]}..."
    
    return answer, idxs

In [49]:
rag_examples = []

questions = [
    "Что такое Python?",
    "Что такое список?",
    "Как работают исключения?"
]

for q in questions:
    ans, idxs = mini_rag(q)
    
    rag_examples.append({
        "question": q,
        "answer": ans,
        "retrieved_sources": idxs.tolist()
    })
    
    print("\nВопрос:", q)
    print("Ответ:", ans)


Вопрос: Что такое Python?
Ответ: Ответ на основе контекста: Функции в Python определяются с помощью ключевого слова def. Списки в Python являются изменяемыми структурами данных и могут хранить несколько элементов. Python — это язык программирования, который ис...

Вопрос: Что такое список?
Ответ: Ответ на основе контекста: Списки в Python являются изменяемыми структурами данных и могут хранить несколько элементов. Циклы позволяют выполнять действия несколько раз. Обработка исключений выполняется с помощью конструкции tr...

Вопрос: Как работают исключения?
Ответ: Ответ на основе контекста: Циклы позволяют выполнять действия несколько раз. Виртуальные окружения позволяют изолировать зависимости проекта. Словари хранят данные в формате ключ-значение....


In [50]:
os.makedirs("artifacts", exist_ok=True)

df_eval.to_csv("artifacts/retrieval_eval.csv", index=False)
df_compare.to_csv("artifacts/retrieval_before_after_update.csv", index=False)

pd.DataFrame(rag_examples).to_csv("artifacts/rag_examples.csv", index=False)

In [45]:
df_eval[df_eval["hit_at_k"] == 0]

,query,expected_source,retrieved_sources,hit_at_k


В данном эксперименте отсутствуют ошибки, так как для всех контрольных запросов релевантный документ попадает в топ-k. Это объясняется небольшим размером базы знаний, простой и однозначной формулировкой запросов